In [17]:
from pydantic import BaseModel,Field
from langchain_openai import ChatOpenAI

class RagResponse(BaseModel):
    answer:str = Field(description="The final answer to the user")
    source_id:int= Field(description="the ID of the document user to answer")
    confidence:float = Field(description="Score from 0 to 1")

model = ChatOpenAI(model="gpt-4o")
#handle structured
structured_llm=model.with_structured_output(RagResponse)

result = structured_llm.invoke("base on doc 42,the price is $10.")
print(result.answer)
print("\n")
print(result.source_id)
print("\n")
print(result.confidence)


The price based on document 42 is $10.


42


0.95


In [18]:
from langchain_core.prompts import ChatPromptTemplate

# 1. RAG Prompt
rag_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a Retrieval-Augmented Generation (RAG) assistant.

Use ONLY the provided DOCUMENT CONTEXT to answer the question.

Rules:
- Do not use your own knowledge.
- Do not invent names, roles, or numbers.
- Only include people explicitly supported by the DOCUMENT CONTEXT.
- If the requested information is not found, return empty lists and number 0.
- The "roles" field should contain the role being asked about.
- The "names" field should contain the names of people who have that role.
- The "number" field should contain the total number of matching people.
"""
    ),
    (
        "user",
        """
DOCUMENT CONTEXT:
{context}

QUESTION:
{question}
"""
    )
])

# 2. Structured Output
class ResponseFormat(BaseModel):
    roles: list[str] = Field(
        description="The role or roles requested by the user"
    )
    
    names: list[str] = Field(
        description="Names of people matching the requested role"
    )
    
    number: int = Field(
        description="Total number of people matching the requested role"
    )

# 3. LLM
model = ChatOpenAI(
    model="gpt-4o",
    temperature=0
)

#for this part, its a combination of (model | parser)
structured_llm = model.with_structured_output(ResponseFormat)


# 4. Build the chain
rag_chain = rag_template | structured_llm

# Simulating a RAG retrieval step
context_data =  """
Once upon a time, in the heart of the Whispering Woods, lay the ancient Kingdom of Old. It was a peaceful realm where towering stone castles carried centuries of history, golden fields stretched across the countryside, and rivers shimmered like liquid silver beneath the sunlight.

The kingdom was ruled by King Aldric, a wise and courageous ruler, and his gentle queen, Queen Aurelia. Queen Aurelia was the guardian of a mysterious magical artifact known as the Chronos Stone, an ancient relic with the power to keep the kingdom perfectly balanced between the past and the present.

Although the Kingdom of Old was small, its ten inhabitants worked together to keep their home safe and prosperous.

King Aldric was the ruler of the Kingdom of Old. He was responsible for making important decisions and ensuring the safety and prosperity of his people.

Queen Aurelia was the kind and wise queen who protected the Chronos Stone and advised King Aldric on matters concerning the kingdom.

Three skilled farmers provided food for the kingdom:

Rowan Greenfield was a hardworking farmer who grew wheat, barley, and vegetables.

Elara Meadow was an expert in herbs, fruits, and medicinal plants.

Tomas Hillcrest was responsible for livestock and maintaining the kingdom's farmland.

Two experienced hunters protected the surrounding wilderness and provided additional food:

Kael Thornwood was a skilled tracker who knew every hidden path within the Whispering Woods.

Lyra Swiftarrow was an expert archer who hunted animals and watched for dangers beyond the kingdom's borders.

Three loyal knights protected the kingdom:

Sir Cedric Ironshield was captain of the royal knights and commander of the kingdom's defenses.

Sir Gareth Stormblade was a fearless warrior responsible for guarding the castle and its gates.

Dame Seraphina Dawn was a highly skilled knight entrusted with protecting Queen Aurelia and the Chronos Stone.

Together, these ten people formed the heart of the Kingdom of Old:

1 King + 1 Queen + 3 Farmers + 2 Hunters + 3 Knights = 10 people.

For generations, they lived in harmony. The farmers sustained the kingdom, the hunters watched over the forests, the knights defended its walls, and the King and Queen maintained peace throughout the land.

But at the center of everything remained the Chronos Stone.

As long as the stone remained protected, the Kingdom of Old would continue to exist in perfect harmony between past and present.

Or so they believed.
"""
     
# 5. Ask question
query = "How many Hunters are in the Old Kingdom?"

result = rag_chain.invoke({
    "context": context_data,
    "question": query
})

print(result)


roles=['Hunters'] names=['Kael Thornwood', 'Lyra Swiftarrow'] number=2


In [20]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI


# ============================================================
# 1. PROMPT
# ============================================================

rag_template = ChatPromptTemplate.from_messages([
    (
        "system",
        """
You are a Retrieval-Augmented Generation (RAG) assistant.

Use ONLY the provided DOCUMENT CONTEXT to answer the question.

Rules:
- Do not use your own knowledge.
- Do not invent names, roles, or numbers.
- Only include information explicitly supported by the DOCUMENT CONTEXT.
- If the requested information is not found, say:
  "I cannot find this information in the provided context."
"""
    ),
    (
        "user",
        """
DOCUMENT CONTEXT:
{context}

QUESTION:
{question}
"""
    )
])


# ============================================================
# 2. MODEL
# ============================================================

model = ChatOpenAI(
    model="gpt-4o",
    temperature=0
)


# ============================================================
# 3. PARSER
# ============================================================

parser = StrOutputParser()


# ============================================================
# 4. BUILD CHAIN
# prompt | model | parser
# ============================================================

rag_chain = rag_template | model | parser


# ============================================================
# 5. SIMULATING RAG RETRIEVAL
# ============================================================

context_data = """
Once upon a time, in the heart of the Whispering Woods, lay the
ancient Kingdom of Old.

The kingdom was ruled by King Aldric and Queen Aurelia.

King Aldric was the ruler of the Kingdom of Old.

Queen Aurelia was the guardian of the Chronos Stone.

Three skilled farmers provided food for the kingdom:

Rowan Greenfield was a hardworking farmer who grew wheat,
barley, and vegetables.

Elara Meadow was an expert in herbs, fruits, and medicinal plants.

Tomas Hillcrest was responsible for livestock and maintaining
the kingdom's farmland.

Two experienced hunters protected the surrounding wilderness:

Kael Thornwood was a skilled tracker who knew every hidden path
within the Whispering Woods.

Lyra Swiftarrow was an expert archer who hunted animals and
watched for dangers beyond the kingdom's borders.

Three loyal knights protected the kingdom:

Sir Cedric Ironshield was captain of the royal knights.

Sir Gareth Stormblade was responsible for guarding the castle.

Dame Seraphina Dawn was entrusted with protecting Queen Aurelia
and the Chronos Stone.

Together, these ten people formed the heart of the Kingdom of Old:

1 King + 1 Queen + 3 Farmers + 2 Hunters + 3 Knights = 10 people.
"""


# ============================================================
# 6. QUESTION
# ============================================================

query = "How many Hunters are in the Old Kingdom?"


# ============================================================
# 7. INVOKE CHAIN
# ============================================================

result = rag_chain.invoke({
    "context": context_data,
    "question": query
})


print(result)

There are 2 Hunters in the Kingdom of Old.
